# EEG Single Instance Overfitting Verification

This notebook verifies that the model successfully overfits a single EEG segment.

**Configuration:**
- EEG shape: (1, 2, 600) - 1 channel, 2 electrodes, 600 time samples
- Patch size: (2, 10) - 60 total patches

**Expected behavior:**
- Generated signals should closely match the target
- MSE between generated and target should be very low (<0.01)
- Visual comparison should show near-identical waveforms

## Setup

### Training command:
```bash
python train_eeg_single.py --max_steps 5000
```

Or with your own EEG data:
```bash
python train_eeg_single.py --data_path path/to/eeg.npy --max_steps 5000
```

In [ ]:
# Navigate to PixNerd folder
import os
import sys

NOTEBOOK_DIR = os.getcwd()
print(f"Starting directory: {NOTEBOOK_DIR}")

# Navigate to PixNerd folder (where src/ lives)
PIXNERD_DIR = os.path.join(NOTEBOOK_DIR, "PixNerd")
if os.path.exists(PIXNERD_DIR):
    os.chdir(PIXNERD_DIR)
    print(f"Changed to: {os.getcwd()}")
elif os.path.basename(NOTEBOOK_DIR) == "PixNerd":
    print(f"Already in PixNerd directory: {NOTEBOOK_DIR}")
else:
    parent = os.path.dirname(NOTEBOOK_DIR)
    pixnerd_in_parent = os.path.join(parent, "PixNerd")
    if os.path.exists(pixnerd_in_parent):
        os.chdir(pixnerd_in_parent)
        print(f"Changed to: {os.getcwd()}")
    else:
        print(f"WARNING: Could not find PixNerd folder")

if os.path.exists("src"):
    print("Found src/ directory")
else:
    print("ERROR: src/ directory not found!")

In [ ]:
from pathlib import Path
import json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Paths
PIXNERD_ROOT = Path(os.getcwd())

# ============================================================
# CHECKPOINT PATH - UPDATE THIS TO YOUR TRAINED MODEL
# ============================================================
EXP_DIR = PIXNERD_ROOT / "workdirs" / "exp_eeg_single_overfit"
CKPT_PATH = EXP_DIR / "checkpoints" / "last.ckpt"
TARGET_SIGNAL_PATH = EXP_DIR / "target_signal.npy"
CONFIG_PATH = EXP_DIR / "config.json"
# ============================================================

OUTPUT_DIR = PIXNERD_ROOT / "outputs" / "eeg_single"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Checkpoint path: {CKPT_PATH}")
print(f"Checkpoint exists: {CKPT_PATH.exists()}")
print(f"Target signal exists: {TARGET_SIGNAL_PATH.exists()}")
print(f"Config exists: {CONFIG_PATH.exists()}")
print(f"Device: {DEVICE}")

## Load Configuration and Target Signal

In [ ]:
# Load config
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        config = json.load(f)
    print("Loaded config:")
    for k, v in config.items():
        print(f"  {k}: {v}")
else:
    # Default config
    config = {
        "num_channels": 2,
        "num_samples": 600,
        "patch_size_h": 2,
        "patch_size_w": 10,
        "hidden_size": 256,
        "decoder_hidden_size": 32,
        "num_encoder_blocks": 6,
        "num_decoder_blocks": 2,
        "num_groups": 8,
    }
    print("Using default config")

# Extract config
NUM_CHANNELS = config["num_channels"]
NUM_SAMPLES = config["num_samples"]
PATCH_SIZE_H = config["patch_size_h"]
PATCH_SIZE_W = config["patch_size_w"]
HIDDEN_SIZE = config["hidden_size"]
DECODER_HIDDEN_SIZE = config["decoder_hidden_size"]
NUM_ENCODER_BLOCKS = config["num_encoder_blocks"]
NUM_DECODER_BLOCKS = config["num_decoder_blocks"]
NUM_GROUPS = config["num_groups"]

num_patches = (NUM_CHANNELS // PATCH_SIZE_H) * (NUM_SAMPLES // PATCH_SIZE_W)
print(f"\nTotal patches: {num_patches}")

In [ ]:
# Load target signal
target_signal = np.load(TARGET_SIGNAL_PATH)
target_tensor = torch.from_numpy(target_signal).float()

print(f"Target signal shape: {target_signal.shape}")
print(f"Target range: [{target_signal.min():.3f}, {target_signal.max():.3f}]")

# Visualize target signal
fig, axes = plt.subplots(NUM_CHANNELS, 1, figsize=(14, 3 * NUM_CHANNELS))
if NUM_CHANNELS == 1:
    axes = [axes]

time = np.arange(NUM_SAMPLES) / 200.0  # Assuming 200 Hz sampling rate

for ch in range(NUM_CHANNELS):
    axes[ch].plot(time, target_signal[0, ch, :], 'b-', linewidth=0.8)
    axes[ch].set_ylabel(f"Channel {ch}")
    axes[ch].set_xlabel("Time (s)")
    axes[ch].set_xlim([0, time[-1]])
    axes[ch].grid(True, alpha=0.3)

plt.suptitle("Target EEG Signal (to be memorized)", fontsize=14)
plt.tight_layout()
plt.show()

## Build Model

In [ ]:
# Import PixNerd components
from src.models.autoencoder.pixel import PixelAE
from src.models.conditioner.class_label import LabelConditioner
from src.models.transformer.pixnerd_eeg_heavydecoder import PixNerDiT_EEG
from src.diffusion.flow_matching.scheduling import LinearScheduler
from src.diffusion.flow_matching.sampling import EulerSampler, ode_step_fn
from src.diffusion.base.guidance import simple_guidance_fn
from src.diffusion.flow_matching.training import FlowMatchingTrainer
from src.callbacks.simple_ema import SimpleEMA
from src.lightning_model import LightningModel

print("Imports successful!")

In [ ]:
print("Initializing model components...")

main_scheduler = LinearScheduler()

vae = PixelAE(scale=1.0)

conditioner = LabelConditioner(num_classes=1)

denoiser = PixNerDiT_EEG(
    in_channels=1,
    patch_size_h=PATCH_SIZE_H,
    patch_size_w=PATCH_SIZE_W,
    num_groups=NUM_GROUPS,
    hidden_size=HIDDEN_SIZE,
    decoder_hidden_size=DECODER_HIDDEN_SIZE,
    num_encoder_blocks=NUM_ENCODER_BLOCKS,
    num_decoder_blocks=NUM_DECODER_BLOCKS,
    num_classes=1,
)

# Sampler with no guidance for overfitting
sampler = EulerSampler(
    num_steps=50,
    guidance=1.0,
    guidance_interval_min=0.0,
    guidance_interval_max=1.0,
    scheduler=main_scheduler,
    w_scheduler=LinearScheduler(),
    guidance_fn=simple_guidance_fn,
    step_fn=ode_step_fn,
)

trainer_stub = FlowMatchingTrainer(
    scheduler=main_scheduler,
    lognorm_t=True,
    timeshift=1.0,
)

ema_tracker = SimpleEMA(decay=0.9999)

model = LightningModel(
    vae=vae,
    conditioner=conditioner,
    denoiser=denoiser,
    diffusion_trainer=trainer_stub,
    diffusion_sampler=sampler,
    ema_tracker=ema_tracker,
    optimizer=None,
    lr_scheduler=None,
    eval_original_model=False,
)

model.eval()
model.to(DEVICE)
print(f"Model initialized and moved to {DEVICE}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Load Checkpoint

In [ ]:
print(f"Loading checkpoint from: {CKPT_PATH}")
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
missing, unexpected = model.load_state_dict(ckpt["state_dict"], strict=False)
print(f"Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}")
print("Checkpoint loaded successfully!")

## Helper Functions

In [ ]:
@torch.no_grad()
def sample_eeg(
    num_samples: int = 1,
    seed: int = 42,
    num_steps: int = 50,
    guidance: float = 1.0,
):
    """Generate EEG samples."""
    torch.manual_seed(seed)
    
    # Configure sampler
    model.diffusion_sampler.guidance = guidance
    model.diffusion_sampler.num_steps = num_steps
    
    # Generate noise with correct shape
    noise = torch.randn(num_samples, 1, NUM_CHANNELS, NUM_SAMPLES, device=DEVICE)
    
    # Get condition (class 0)
    labels = [0] * num_samples
    condition, uncondition = model.conditioner(labels)
    condition = condition.to(DEVICE)
    uncondition = uncondition.to(DEVICE)
    
    # Sample
    samples = model.diffusion_sampler(
        model.ema_denoiser,
        noise,
        condition,
        uncondition,
    )
    
    # Decode (identity for PixelAE)
    signals = model.vae.decode(samples)
    signals = torch.clamp(signals, -1.0, 1.0)
    
    return signals.cpu()


def compute_metrics(generated, target):
    """Compute MSE and correlation between generated and target signals."""
    mse = F.mse_loss(generated, target).item()
    
    # Flatten and compute correlation
    gen_flat = generated.flatten().numpy()
    tgt_flat = target.flatten().numpy()
    correlation = np.corrcoef(gen_flat, tgt_flat)[0, 1]
    
    return mse, correlation


def plot_comparison(target, generated, title=""):
    """Plot comparison between target and generated signals."""
    time = np.arange(NUM_SAMPLES) / 200.0
    
    fig, axes = plt.subplots(NUM_CHANNELS, 1, figsize=(14, 3 * NUM_CHANNELS))
    if NUM_CHANNELS == 1:
        axes = [axes]
    
    for ch in range(NUM_CHANNELS):
        axes[ch].plot(time, target[0, ch, :], 'b-', linewidth=1.0, label='Target', alpha=0.7)
        axes[ch].plot(time, generated[0, ch, :], 'r--', linewidth=1.0, label='Generated', alpha=0.7)
        axes[ch].set_ylabel(f"Channel {ch}")
        axes[ch].set_xlabel("Time (s)")
        axes[ch].set_xlim([0, time[-1]])
        axes[ch].legend(loc='upper right')
        axes[ch].grid(True, alpha=0.3)
    
    if title:
        plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    return fig


print("Helper functions defined.")

## Generate and Compare

Generate samples with different seeds and compare to target.

In [ ]:
# Generate multiple samples
num_test_samples = 5
all_samples = []
all_mses = []
all_corrs = []

for seed in range(num_test_samples):
    samples = sample_eeg(
        num_samples=1,
        seed=seed,
        num_steps=100,
        guidance=1.0,
    )
    all_samples.append(samples[0])
    
    # Compute metrics
    mse, corr = compute_metrics(samples[0], target_tensor)
    all_mses.append(mse)
    all_corrs.append(corr)
    print(f"Seed {seed}: MSE={mse:.6f}, Correlation={corr:.4f}")

print(f"\nAverage MSE: {np.mean(all_mses):.6f}")
print(f"Average Correlation: {np.mean(all_corrs):.4f}")

In [ ]:
# Visualize best sample
best_idx = np.argmin(all_mses)
best_sample = all_samples[best_idx]

fig = plot_comparison(
    target_signal,
    best_sample.numpy(),
    title=f"Best Sample (seed={best_idx}, MSE={all_mses[best_idx]:.6f}, r={all_corrs[best_idx]:.4f})"
)
plt.savefig(OUTPUT_DIR / "overfitting_comparison.png", dpi=150)
plt.show()

In [ ]:
# Visualize all samples overlaid
time = np.arange(NUM_SAMPLES) / 200.0

fig, axes = plt.subplots(NUM_CHANNELS, 1, figsize=(14, 3 * NUM_CHANNELS))
if NUM_CHANNELS == 1:
    axes = [axes]

for ch in range(NUM_CHANNELS):
    # Plot target
    axes[ch].plot(time, target_signal[0, ch, :], 'k-', linewidth=2.0, label='Target', zorder=10)
    
    # Plot all generated samples
    for i, sample in enumerate(all_samples):
        sample_np = sample.numpy()
        label = f'Seed {i}' if i == 0 else None
        axes[ch].plot(time, sample_np[0, ch, :], '--', linewidth=0.8, alpha=0.5, label=f'Seed {i}')
    
    axes[ch].set_ylabel(f"Channel {ch}")
    axes[ch].set_xlabel("Time (s)")
    axes[ch].set_xlim([0, time[-1]])
    axes[ch].legend(loc='upper right', fontsize=8)
    axes[ch].grid(True, alpha=0.3)

plt.suptitle("All Generated Samples vs Target", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "all_samples_comparison.png", dpi=150)
plt.show()

## Difference Analysis

In [ ]:
# Analyze differences
best_sample_np = best_sample.numpy()
diff = np.abs(best_sample_np - target_signal)

fig, axes = plt.subplots(NUM_CHANNELS, 1, figsize=(14, 3 * NUM_CHANNELS))
if NUM_CHANNELS == 1:
    axes = [axes]

time = np.arange(NUM_SAMPLES) / 200.0

for ch in range(NUM_CHANNELS):
    axes[ch].fill_between(time, 0, diff[0, ch, :], alpha=0.7, color='red')
    axes[ch].set_ylabel(f"Channel {ch}\n|Error|")
    axes[ch].set_xlabel("Time (s)")
    axes[ch].set_xlim([0, time[-1]])
    axes[ch].grid(True, alpha=0.3)
    max_err = diff[0, ch, :].max()
    mean_err = diff[0, ch, :].mean()
    axes[ch].set_title(f"Max error: {max_err:.4f}, Mean error: {mean_err:.4f}")

plt.suptitle("Absolute Error (Best Sample)", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "error_analysis.png", dpi=150)
plt.show()

## Metrics Summary

In [ ]:
print("="*60)
print("EEG OVERFITTING VERIFICATION SUMMARY")
print("="*60)
print(f"Signal shape: (1, {NUM_CHANNELS}, {NUM_SAMPLES})")
print(f"Patch size: ({PATCH_SIZE_H}, {PATCH_SIZE_W})")
print(f"Total patches: {num_patches}")
print()
print(f"Number of samples tested: {num_test_samples}")
print(f"Average MSE: {np.mean(all_mses):.6f}")
print(f"Average Correlation: {np.mean(all_corrs):.4f}")
print(f"Best MSE: {np.min(all_mses):.6f} (seed {np.argmin(all_mses)})")
print(f"Best Correlation: {np.max(all_corrs):.4f} (seed {np.argmax(all_corrs)})")
print()

# Quality assessment
avg_mse = np.mean(all_mses)
avg_corr = np.mean(all_corrs)

if avg_mse < 0.001 and avg_corr > 0.99:
    print("EXCELLENT: Model has perfectly memorized the EEG signal!")
elif avg_mse < 0.01 and avg_corr > 0.95:
    print("GOOD: Model has mostly memorized the EEG signal.")
elif avg_mse < 0.05 and avg_corr > 0.9:
    print("FAIR: Model is learning but needs more training.")
else:
    print("POOR: Model has not yet memorized the signal. Train longer!")

print("="*60)

## Time-Domain Super-Resolution Test (Optional)

Test if the overfitted model can generate at higher temporal resolution.

In [ ]:
def set_decoder_scale(scale_h: float, scale_w: float):
    """Set decoder patch scaling for super-resolution."""
    for net in [model.denoiser, getattr(model, "ema_denoiser", None)]:
        if net is None:
            continue
        net.decoder_patch_scaling_h = scale_h
        net.decoder_patch_scaling_w = scale_w


@torch.no_grad()
def sample_superres(
    height: int,
    width: int,
    seed: int = 42,
    num_steps: int = 50,
):
    """Generate super-resolution sample."""
    torch.manual_seed(seed)
    
    scale_h = height / NUM_CHANNELS
    scale_w = width / NUM_SAMPLES
    set_decoder_scale(scale_h, scale_w)
    
    model.diffusion_sampler.guidance = 1.0
    model.diffusion_sampler.num_steps = num_steps
    
    noise = torch.randn(1, 1, height, width, device=DEVICE)
    
    condition, uncondition = model.conditioner([0])
    condition = condition.to(DEVICE)
    uncondition = uncondition.to(DEVICE)
    
    samples = model.diffusion_sampler(
        model.ema_denoiser,
        noise,
        condition,
        uncondition,
    )
    
    signals = model.vae.decode(samples)
    signals = torch.clamp(signals, -1.0, 1.0)
    
    # Reset scale
    set_decoder_scale(1.0, 1.0)
    
    return signals.cpu()


# Generate at different time resolutions (keep channels same)
print("Generating time-domain super-resolution samples...")

sig_native = best_sample  # Already have this
sig_2x = sample_superres(NUM_CHANNELS, NUM_SAMPLES * 2, seed=best_idx, num_steps=100)  # 1200 samples

# Compare native vs 2x resolution
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

time_native = np.arange(NUM_SAMPLES) / 200.0
time_2x = np.arange(NUM_SAMPLES * 2) / 400.0  # 2x sampling rate

# Native resolution
axes[0].plot(time_native, sig_native[0, 0, :].numpy(), 'b-', linewidth=1.0)
axes[0].set_title(f"Native Resolution ({NUM_SAMPLES} samples)")
axes[0].set_xlabel("Time (s)")
axes[0].set_xlim([0, max(time_native[-1], time_2x[-1])])
axes[0].grid(True, alpha=0.3)

# 2x resolution
axes[1].plot(time_2x, sig_2x[0, 0, 0, :].numpy(), 'r-', linewidth=1.0)
axes[1].set_title(f"2x Super-Resolution ({NUM_SAMPLES * 2} samples)")
axes[1].set_xlabel("Time (s)")
axes[1].set_xlim([0, max(time_native[-1], time_2x[-1])])
axes[1].grid(True, alpha=0.3)

plt.suptitle("Time-Domain Super-Resolution", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "superres_comparison.png", dpi=150)
plt.show()

In [ ]:
print("Done!")
print(f"Outputs saved to: {OUTPUT_DIR}")